# Notebook 05: Predictive Modeling

**Project:** City of Boston – Building & Housing Violations Analysis  
**Group:** Mengxing Wang, Jiacheng He, Xiao Luo, Renwei Li

## Overview

This notebook builds two models:

1. **RandomForest Regression** — Predicts the violation count of a neighborhood based on its normalized rate, population, and other aggregate features. This demonstrates how population and density interact with violation frequency.

2. **KMeans Clustering** — Groups neighborhoods into clusters based on their violation profile (raw count, per-capita rate, year-over-year change). This surfaces natural groupings that may inform targeted policy.

All figures are saved to `figures/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'
FIGURES_DIR = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

# Load datasets
df = pd.read_csv(PROCESSED_DIR / 'violations_clean.csv', low_memory=False)
df['status_dttm'] = pd.to_datetime(df['status_dttm'], errors='coerce')
stats = pd.read_csv(PROCESSED_DIR / 'neighborhood_stats.csv')

print(f'Violations records: {len(df):,}')
print(f'Neighborhood stats rows: {len(stats)}')
stats.head()

## 1. Feature Engineering for Regression

We enrich the neighborhood-level stats table with additional features derived from the full dataset:
- **open_rate**: fraction of violations still open
- **pct_maintenance**: fraction of violations categorized as maintenance
- **pct_unsafe**: fraction classified as unsafe/dangerous
- **year_range**: span of years with violations
- **unique_codes**: diversity of violation types

In [ ]:
df_nbhd = df[df['has_neighborhood'] == True].copy()

agg = df_nbhd.groupby('neighborhood').agg(
    open_rate=('status', lambda x: (x == 'Open').mean()),
    unique_codes=('code', 'nunique'),
    pct_maintenance=('description', lambda x: x.str.contains('Maintenance', case=False, na=False).mean()),
    pct_unsafe=('description', lambda x: x.str.contains('Unsafe|Dangerous', case=False, na=False).mean()),
).reset_index()

# Year range for neighborhoods with date data
df_dated = df_nbhd[df_nbhd['year'].notna()]
year_range = df_dated.groupby('neighborhood')['year'].agg(lambda x: x.max() - x.min()).reset_index()
year_range.columns = ['neighborhood', 'year_range']

# Merge all features
features_df = stats.merge(agg, on='neighborhood', how='left')\
                   .merge(year_range, on='neighborhood', how='left')

# Drop rows without population (can't normalize)
features_df = features_df.dropna(subset=['population_2020', 'violations_per_1000'])
features_df = features_df.fillna(0)
print(f'Feature matrix shape: {features_df.shape}')
features_df[['neighborhood','violation_count','violations_per_1000','open_rate','unique_codes','pct_maintenance','pct_unsafe','year_range']].head()

## 2. RandomForest Regression

### Target
We predict **`violation_count`** (raw count) from population, normalized rate, and the engineered features above.

### Approach
Given the small neighborhood-level dataset (n ≈ 20), we use **5-fold cross-validation** rather than a train/test split to get a robust estimate of generalization error.

In [ ]:
feature_cols = [
    'population_2020', 'violations_per_1000',
    'open_rate', 'unique_codes',
    'pct_maintenance', 'pct_unsafe', 'year_range'
]

X = features_df[feature_cols].values
y = features_df['violation_count'].values

rf = RandomForestRegressor(n_estimators=200, max_depth=4, random_state=42)

cv = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = cross_val_score(rf, X, y, cv=cv, scoring='neg_mean_absolute_error')
r2_scores  = cross_val_score(rf, X, y, cv=cv, scoring='r2')

print(f'Cross-validated MAE:  {-mae_scores.mean():.1f} ± {mae_scores.std():.1f}')
print(f'Cross-validated R²:   {r2_scores.mean():.3f} ± {r2_scores.std():.3f}')

### 2a. Feature Importance

We fit the full model to the entire dataset and inspect feature importances.

In [ ]:
rf.fit(X, y)
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = sns.color_palette('viridis', len(importances))
ax.barh(importances.index[::-1], importances.values[::-1], color=colors)
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)')
ax.set_title('RandomForest Feature Importances\n(Predicting Violation Count)', fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '13_rf_feature_importance.png', bbox_inches='tight')
plt.close()
print('Saved: 13_rf_feature_importance.png')
print(importances)

### 2b. Actual vs. Predicted (Full-fit)

We visualize how closely the model's predictions match actual counts on the training data.

In [ ]:
y_pred = rf.predict(X)
full_mae = mean_absolute_error(y, y_pred)
full_r2  = r2_score(y, y_pred)

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(y, y_pred, s=80, color='steelblue', alpha=0.8, zorder=5)
maxval = max(y.max(), y_pred.max()) * 1.05
ax.plot([0, maxval], [0, maxval], 'r--', linewidth=1.5, label='Perfect fit')

for i, nbhd in enumerate(features_df['neighborhood']):
    ax.annotate(nbhd, (y[i], y_pred[i]), xytext=(5, 3),
                textcoords='offset points', fontsize=7, alpha=0.75)

ax.set_xlabel('Actual Violation Count')
ax.set_ylabel('Predicted Violation Count')
ax.set_title(f'RF Regression: Actual vs. Predicted\nMAE={full_mae:.1f}, R²={full_r2:.3f}', fontsize=12)
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / '14_rf_actual_vs_predicted.png', bbox_inches='tight')
plt.close()
print('Saved: 14_rf_actual_vs_predicted.png')